In [1]:
from pygel3d import hmesh, jupyter_display as jd, gl_display as gl
import numpy as np
import math
from commons.utils import *
from commons.display import *
from preprocessing import voxelize
from medial_axis_formation import deep_points, postprocessing
from medial_axis_processing import unfolding, inverse_apply
from medial_axis_processing.medial_axis import MedialAxis

## Load data

In [2]:
input_mesh_path = '../data/x19_rough_center.ply'

input_mesh = hmesh.load(input_mesh_path)
# jd.display(input_mesh)

## Preprocess data

In [3]:
voxel_size = 1.0
smooth_steps = 5

voxelized = voxelize.voxel_remesh(input_mesh, voxel_size)
smooth(voxelized, smooth_steps)

print(len(voxelized.vertices()))
jd.display(voxelized)

22397


FigureWidget({
    'data': [{'color': '#dddddd',
              'flatshading': False,
              'i': array([15561,     2,     3, ..., 22369, 22339, 21188]),
              'j': array([    0,     3,     2, ..., 21216, 22306, 22257]),
              'k': array([    1,     4,  1528, ..., 22363, 21914, 21208]),
              'type': 'mesh3d',
              'uid': 'f51762b1-a5f7-455b-bbf2-faafc962e499',
              'x': array([-35.81698025, -35.9457938 ,   9.57637353, ...,  -8.76216241,
                           -6.74280573,  -6.20404038]),
              'y': array([-17.76799953, -17.30691355, -26.07324332, ...,  21.17766712,
                           18.00920956,  18.353588  ]),
              'z': array([12.17689001, 12.10248474, -7.34419917, ..., -0.07977205,  0.07721648,
                           1.01351714])},
             {'hoverinfo': 'none',
              'line': {'color': 'rgb(125,0,0)', 'width': 1},
              'mode': 'lines',
              'type': 'scatter3d',
           

## Compute Medial Axis

In [4]:
deep_points_params = { 
    "sigma_p": 5.0,
    "sigma_q": 1.5,
    "sigma_s": 4.5,
    "omega": math.radians(90),
    "curve_regularization_threshold": 0.95,
    "run_sinking_smoothing": True,
    "run_regularization": True
} # ✌︎

outer_points, inner_points = deep_points.deep_points(voxelized, deep_points_params)

# display_result(voxelize, outer_points, inner_points)

Running normal smoothing...
Running sinking of inner points...
Running skeleton formation...
Running curve point regularization...


Extract a single sheet out of the medial sheet (the output of `deep_points.medial_sheet` is two sheets)

In [5]:
dihedral_angle_threshold = 45

medial_sheet = postprocessing.to_medial_sheet(voxelized, inner_points, dihedral_angle_threshold)

jd.display(medial_sheet)

FigureWidget({
    'data': [{'color': '#dddddd',
              'flatshading': False,
              'i': array([    0,  1054,     3, ..., 10461,  9893,  9893]),
              'j': array([    1,     0,  1054, ..., 10460, 10437, 10438]),
              'k': array([    2,     2,     4, ..., 10462, 10438,  9904]),
              'type': 'mesh3d',
              'uid': '3c2a3de1-773f-4df0-aa36-b7e654bdb334',
              'x': array([ 9.67582008,  9.84310755, 10.02186863, ..., -7.44011584, -8.03241399,
                          -7.23213468]),
              'y': array([-25.48027815, -25.06118324, -25.4133095 , ..., -29.35765585,
                           19.10410189,  19.19749228]),
              'z': array([-8.37563126, -8.20813148, -8.28411318, ..., -6.19962945, -0.2862002 ,
                           2.22196252])},
             {'hoverinfo': 'none',
              'line': {'color': 'rgb(125,0,0)', 'width': 1},
              'mode': 'lines',
              'type': 'scatter3d',
              'ui

Create `MedialAxis` object from results to build correspondences between object vertices and vertices on the medial sheet

## Process Medial Axis 

In [6]:
medial_axis = MedialAxis(medial_sheet, inner_points, outer_points)

In [7]:
unfolding.unfold_medial_axis(medial_axis)
inverse_apply.apply_inverse_medial_axis_transform(voxelized, medial_axis)
jd.display(voxelized)

FigureWidget({
    'data': [{'color': '#dddddd',
              'flatshading': False,
              'i': array([15561,     2,     3, ..., 22369, 22339, 21188]),
              'j': array([    0,     3,     2, ..., 21216, 22306, 22257]),
              'k': array([    1,     4,  1528, ..., 22363, 21914, 21208]),
              'type': 'mesh3d',
              'uid': 'e445cca3-de05-4be7-a56f-3ecfe17f3a0e',
              'x': array([61.04983323, 61.36971317, 11.48925565, ..., 57.42748625, 57.44599831,
                          56.27318717]),
              'y': array([-5.14617202e+01, -5.11169383e+01, -3.05928992e+01, ...,  3.40149305e-03,
                          -1.05753079e+00, -8.56103747e-01]),
              'z': array([-0.15594179, -0.4135339 , -1.15301676, ..., -1.89650214,  1.57798866,
                           1.55222599])},
             {'hoverinfo': 'none',
              'line': {'color': 'rgb(125,0,0)', 'width': 1},
              'mode': 'lines',
              'type': 'scatter3d',